In [234]:
pip install -U kubeflow-training


[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [235]:
def train_func():
    import os
    import logging
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        TrainingArguments,
        DataCollatorForLanguageModeling,
        pipeline,
    )
    from trl import SFTTrainer, SFTConfig
    from datasets import load_dataset
    from datasets.distributed import split_dataset_by_node
    from peft import LoraConfig, get_peft_model
    from accelerate import PartialState

    rank = int(os.environ["RANK"])

    # Create system prompt
    system_message = "Solve the given high school math problem. Provide short answer."

    model_name = "meta-llama/Llama-3.2-1B-Instruct"
    model = AutoModelForCausalLM.from_pretrained(
        pretrained_model_name_or_path=model_name,
    )
    tokenizer = AutoTokenizer.from_pretrained(
        pretrained_model_name_or_path=model_name,
    )
    tokenizer.pad_token = tokenizer.eos_token

    def format_dataset(example):
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": example['question']},
            {"role": "assistant", "content": example['answer']}
        ]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=False
        )
        return {"prompt": prompt}

    # Use pipeline to retrieve sample response for test sample, to visually review training progress
    def infer_answer(model, question):
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": question},
        ]
        pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0, temperature = 0.1)
        prompt = pipe.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        outputs = pipe(messages, max_new_tokens=256)
        return outputs[0]['generated_text'][-1]['content']

    dataset = load_dataset("openai/gsm8k", "main")
    train_data = dataset["train"].map(format_dataset, remove_columns=['question', 'answer'])
    eval_data = dataset["test"].map(format_dataset, remove_columns=['question', 'answer'])

    training_args = SFTConfig(
        dataset_text_field="prompt",
        max_seq_length=1024,
        output_dir="/tmp",
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=1,
        learning_rate=1e-6,
        logging_dir="/logs",
        eval_strategy="epoch",
        save_strategy="no",
        fsdp="full_shard",
        fsdp_config={
            "fsdp_state_dict_type": "SHARDED_STATE_DICT",
            "fsdp_sharding_strategy": "FULL_SHARD",
        },
    )

    trainer = SFTTrainer(
        model=model,
        train_dataset=train_data,
        eval_dataset=eval_data,
        tokenizer=tokenizer,
        args=training_args,
    )

    # Check model response for sample test data to see how untrained model responds
    if rank == 0:
        test_prompt_answer = infer_answer(model, dataset['test']['question'][0])
        print(f"Query:\n{dataset['test']['question'][0]}")
        print(f"Original Answer:\n{dataset['test']['answer'][0]}")
        print(f"Generated Answer:\n{test_prompt_answer}")

    # Train and save the model.
    trainer.train()

    # https://github.com/huggingface/transformers/issues/30491
    if trainer.is_fsdp_enabled:
        trainer.accelerator.state.fsdp_plugin.set_state_dict_type("FULL_STATE_DICT")
    trainer.save_model("./saved-model")

    # TODO remove before merging to examples
    print("parallel_mode: " + str(trainer.args.parallel_mode))
    print("is_model_parallel: " + str(trainer.is_model_parallel))
    print("model_wrapped: " + str(trainer.model_wrapped))

    # Check model response for sample test data to see how trained model responds
    if rank == 0:
        model = AutoModelForCausalLM.from_pretrained(
            pretrained_model_name_or_path="./saved-model/",
        )
        test_prompt_answer = infer_answer(model, dataset['test']['question'][0])
        print(f"Query:\n{dataset['test']['question'][0]}")
        print(f"Original Answer:\n{dataset['test']['answer'][0]}")
        print(f"Generated Answer:\n{test_prompt_answer}")

In [238]:
from kubeflow.training import TrainingClient
from kubernetes import client
from kubernetes.client import (
    V1EnvVar,
    V1EnvVarSource,
    V1SecretKeySelector
)

job_name = "pytorch-fsdp"

# Provide URL and token with all needed rights
# On OpenShift, you can retrieve the token by running `oc whoami -t`,
# and the server with `oc cluster-info`.

# token = ""
# openshift_api_url = ""

# api_key = {"authorization": "Bearer " + token}
# config = client.Configuration(host=openshift_api_url, api_key=api_key)
# config.verify_ssl = False
# tc = TrainingClient(client_configuration=config)


# Alternatively add edit role for user running this Notebook using oc CLI:
# oc adm policy add-role-to-user edit system:serviceaccount:<namespace>:<workbench name> -n <namespace>
tc = TrainingClient()

tc.create_job(
    job_kind="PyTorchJob",
    name=job_name,
    train_func=train_func,
    num_workers=2,
    num_procs_per_worker="auto",
    resources_per_worker={"gpu": 1},
    base_image="quay.io/modh/training:py311-cuda121-torch241",
    env_vars=[
        V1EnvVar(name="HF_TOKEN", value_from=V1EnvVarSource(secret_key_ref=V1SecretKeySelector(key="HF_TOKEN", name="hf-token"))),
        V1EnvVar(name="NCCL_DEBUG", value="INFO"),
#        V1EnvVar(name="TOKENIZERS_PARALLELISM", value="false"),
    ],
)

RuntimeError: Failed to create PyTorchJob: ksuta-fsdp/pytorch-fsdp

In [ ]:
logs, _ = tc.get_job_logs(job_name, follow=True)

Map: 100%|██████████| 1319/1319 [00:00<00:00, 13056.68 examples/s]
[Pod pytorch-fsdp-master-0]: pytorch-fsdp-master-0:72:72 [0] NCCL INFO Bootstrap : Using eth0:10.128.4.132<0>
[Pod pytorch-fsdp-master-0]: pytorch-fsdp-master-0:72:72 [0] NCCL INFO NET/Plugin : dlerror=libnccl-net.so: cannot open shared object file: No such file or directory No plugin found (libnccl-net.so), using internal implementation
[Pod pytorch-fsdp-master-0]: pytorch-fsdp-master-0:72:72 [0] NCCL INFO cudaDriverVersion 12040
[Pod pytorch-fsdp-master-0]: NCCL version 2.20.5+cuda12.4
[Pod pytorch-fsdp-master-0]: pytorch-fsdp-master-0:72:274 [0] NCCL INFO Failed to open libibverbs.so[.1]
[Pod pytorch-fsdp-master-0]: pytorch-fsdp-master-0:72:274 [0] NCCL INFO NET/Socket : Using [0]eth0:10.128.4.132<0>
[Pod pytorch-fsdp-master-0]: pytorch-fsdp-master-0:72:274 [0] NCCL INFO Using non-device net plugin version 0
[Pod pytorch-fsdp-master-0]: pytorch-fsdp-master-0:72:274 [0] NCCL INFO Using network Socket
[Pod pytorch-fsdp

In [ ]:
tc.delete_job(name=job_name)